In [22]:
import torch
import sys
sys.path.insert(0, '/rsrch5/home/trans_mol_path/cercan/code/CellViT-plus-plus')
from cellvit.models.cell_segmentation.cellvit_256 import CellViT256
from cellvit.utils.tools import close_logger, unflatten_dict
from cellvit.models.classifier.linear_classifier import LinearClassifier
from cellvit.training.experiments.experiment_cell_classifier import (
    ExperimentCellVitClassifier,
)

2025-05-01 21:17:27,137	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
model_path= "/rsrch5/home/trans_mol_path/cercan/code/CellViT-plus-plus/checkpoints/CellViT-256-x40-AMP.pth"
checkpoint_path = "/rsrch5/home/trans_mol_path/cercan/code/CellViT-plus-plus/checkpoints/classifier/sam-h/consep.pth"

In [4]:
model_checkpoint = torch.load(model_path, map_location="cpu")

In [6]:
model_checkpoint["arch"]

'CellViT256'

In [5]:
run_conf = unflatten_dict(model_checkpoint["config"], ".")
run_conf

{'gpu': 0,
 'data': {'dataset': 'PanNuke',
  'num_nuclei_classes': 6,
  'num_tissue_classes': 19},
 'model': {'backbone': 'ViT256'},
 'training': {'drop_rate': 0,
  'attn_drop_rate': 0.1,
  'drop_path_rate': 0.1,
  'mixed_precision': True,
  'eval_every': 20},
 'transformations': {'randomsizedcrop': {'p': 0.1},
  'normalize': {'mean': [0.5, 0.5, 0.5], 'std': [0.5, 0.5, 0.5]}},
 'eval_checkpoint': 'latest_checkpoint.pth',
 'dataset_config': {'tissue_types': {'Adrenal_gland': 0,
   'Bile-duct': 1,
   'Bladder': 2,
   'Breast': 3,
   'Cervix': 4,
   'Colon': 5,
   'Esophagus': 6,
   'HeadNeck': 7,
   'Kidney': 8,
   'Liver': 9,
   'Lung': 10,
   'Ovarian': 11,
   'Pancreatic': 12,
   'Prostate': 13,
   'Skin': 14,
   'Stomach': 15,
   'Testis': 16,
   'Thyroid': 17,
   'Uterus': 18},
  'nuclei_types': {'Background': 0,
   'Neoplastic': 1,
   'Inflammatory': 2,
   'Connective': 3,
   'Dead': 4,
   'Epithelial': 5}}}

In [12]:
model = CellViT256(
                model256_path=None,
                num_nuclei_classes=run_conf["data"]["num_nuclei_classes"],
                num_tissue_classes=run_conf["data"]["num_tissue_classes"],
                regression_loss=run_conf["model"].get("regression_loss", False),
            )

In [15]:
model.load_state_dict(model_checkpoint["model_state_dict"])

<All keys matched successfully>

In [21]:
classifier_path = "/rsrch5/home/trans_mol_path/cercan/code/CellViT-plus-plus/checkpoints/classifier/sam-h/lizard.pth"
classifier_model_checkpoint = torch.load(classifier_path, map_location="cpu")
classifier_run_conf = unflatten_dict(classifier_model_checkpoint["config"], ".")
classifier = LinearClassifier(
                embed_dim=classifier_model_checkpoint["model_state_dict"]["fc1.weight"].shape[1],
                hidden_dim=classifier_run_conf["model"].get("hidden_dim", 100),
                num_classes=classifier_run_conf["data"]["num_classes"],
                drop_rate=0,
            )
classifier.load_state_dict(classifier_model_checkpoint["model_state_dict"])

<All keys matched successfully>

In [30]:
print(classifier)

LinearClassifier(
  (fc1): Linear(in_features=1280, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=6, bias=True)
  (activation): ReLU()
  (dropout): Dropout(p=0, inplace=False)
)


In [31]:
import torch.nn as nn
new_num_classes=5
new_fc2 = nn.Linear(classifier.fc2.in_features, new_num_classes)
classifier.fc2 = new_fc2
print(classifier)

LinearClassifier(
  (fc1): Linear(in_features=1280, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=5, bias=True)
  (activation): ReLU()
  (dropout): Dropout(p=0, inplace=False)
)


In [23]:
classifier_run_conf = unflatten_dict(classifier_model_checkpoint["config"], ".")
classifier_run_conf

{'data': {'dataset': 'lizard_preextracted',
  'normalize_stains_train': False,
  'normalize_stains_val': False,
  'num_classes': 6,
  'label_map': {'0': 'Neutrophil',
   '1': 'Epithelial',
   '2': 'Lymphocyte',
   '3': 'Plasma',
   '4': 'Eosinophil',
   '5': 'Connective tissue'},
  'network_name': 'SAM-H'},
 'model': {'hidden_dim': 512},
 'training': {'weighted_sampling': True,
  'mixed_precision': True,
  'weight_list': [103.09, 2.02, 4.88, 17.39, 136.99, 4.41],
  'scheduler': {'scheduler_type': 'exponential'}},
 'just_load_model': False}

In [26]:
print(classifier_run_conf["training"].get("drop_rate"))

None


In [17]:
model_conf["data"]["num_classes"]

4

In [14]:

model = CellViTSAM(
    model_path=None,
    num_nuclei_classes=model_conf["data"]["num_nuclei_classes"],
    num_tissue_classes=model_conf["data"]["num_tissue_classes"],
    vit_structure=model_conf["model"]["backbone"],
    regression_loss=model_conf["model"].get("regression_loss", False),
)

KeyError: 'num_nuclei_classes'